# Credit Card Fraud Detection - Comparative study for XGboost and AdaBoost

## Preaparing data for the model implementation


In [8]:
# imports 
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [9]:
# load the dataset 
data = pd.read_csv("creditcard.csv")

Before training the model you should split train in fair way so we can really explore the power of using each model 

1- Start by separte the target from data , so the model learn without having answers 

In [10]:
X = data.drop('Class', axis=1)
y = data['Class']

2- Train / Test split

In [11]:
X_train , X_test , y_train , y_test = train_test_split(
    X , y , 
    test_size=0.2,
    random_state = 42,
    stratify = y , # ensures that the class distribution in your training and test sets is the same as in your original dataset
)

3 - Scaling after split the data , we already from v1 tp v28 a are standarised we add also Amount and Time 

In [12]:
scaler = StandardScaler() 
X_train[['Amount','Time']] = scaler.fit_transform(X_train[['Amount','Time']])
X_test[['Amount','Time']] = scaler.transform(X_test[['Amount','Time']])

4 - Handle class imbalanced 
La technique SMOTE (Synthetic Minority Over-sampling Technique) résout le problème des données déséquilibrées en générant des exemples synthétiques pour la classe minoritaire plutôt qu'en dupliquant simplement les exemples existants

In [13]:
from imblearn.over_sampling import SMOTE 
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [14]:
print("AVANT SMOTE:")
print(y_train.value_counts())
print("\nAPRÈS SMOTE:")
print(y_train_resampled.value_counts())

AVANT SMOTE:
Class
0    227451
1       394
Name: count, dtype: int64

APRÈS SMOTE:
Class
0    227451
1     22745
Name: count, dtype: int64


## Model implementation

In [15]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

In [16]:
base = DecisionTreeClassifier(max_depth=1, class_weight='balanced') 

In [17]:
ada = AdaBoostClassifier(
    estimator=base, # c'est un arbre de décision à 1 niveau definit dans la variable base 
    n_estimators=100, #Nombre d'itérations
    learning_rate=0.5, # plus petite valeur = apprentissage plus lent mais potentiellement plus robuste
    random_state=42 # permet d'obtenir exactement les mêmes résultats à chaque exécution
)

In [18]:
ada.fit(X_train_resampled, y_train_resampled) 

,estimator,"DecisionTreeC..., max_depth=1)"
,n_estimators,100
,learning_rate,0.5
,algorithm,'deprecated'
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


XGboost model implementation

In [19]:
from xgboost import XGBClassifier

In [20]:
xgb = XGBClassifier(
    n_estimators=200, # nombre d'arbre
    max_depth=5, # profondeur maximale de chaque arbre
    learning_rate=0.1, # taux d'apprentissage
    eval_metric='logloss', # metriques d'évolution
    random_state=42 # grain aleatoire 
)

xgb.fit(X_train_resampled, y_train_resampled)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


Prediction

In [21]:
y_pred_ada = ada.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

Métriques d'evaluation

Precision : d'apres ce que j'ai predis positif , combien était vraiment positif 

Recall (rappel / sensibilité) : vrais positifs / (vrais positif + faux négatifs)

F1-score : F1 = 2 × (Precision × Recall) / (Precision + Recall)

In [23]:
print("AdaBoost:\n", classification_report(y_test, y_pred_ada))
print("XGBoost:\n", classification_report(y_test, y_pred_xgb))

AdaBoost:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99     56864
           1       0.11      0.88      0.20        98

    accuracy                           0.99     56962
   macro avg       0.56      0.93      0.60     56962
weighted avg       1.00      0.99      0.99     56962

XGBoost:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.74      0.86      0.80        98

    accuracy                           1.00     56962
   macro avg       0.87      0.93      0.90     56962
weighted avg       1.00      1.00      1.00     56962



On remarque clairement qu'adaboost fait trop de faux positif 0.11 de precision (11%) et recall 0.20 (20%).En revanche,le xgboost a donné des résultats très cohérent , et la performence de xgboost et ses performances sont bien meilleures qu'adaboost

ça reste que ces résultats peuve etre améliorer plus car on autiliser des paramétres par defaut et basiques.

Funtuning the model :

On propose des paramétres et on utilise soit GridSearchCV or RandomizedSearch pour avoir la meilleur combinaison 

1- creer le pipline de model 

2- Grid or randomize combinaison 

3- fit le output de l'etape 2 

In [24]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

xgboost

In [25]:
from xgboost import XGBClassifier

In [26]:
xgb_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.1, random_state=42)),
    ('model', XGBClassifier(
        eval_metric='logloss' , 
        random_state=42
    ))
])

In [29]:
#pipeline à tuner 
param_grid_xgb = {
    'model__n_estimators' : [100,200,300] ,
    'model__max_depth' : [3,5,7] , 
    'model__learning_rate' : [0.01,0.05,0.1] ,
    'model__subsample':[0.8,1] , 
    'model__colsample_bytree' :  [0.8,1]
}

In [30]:
#RandomizedSearch
from sklearn.model_selection import RandomizedSearchCV

random_xgb = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_grid_xgb,
    n_iter=15, # le nombre de test
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1, 
    random_state=42    
)
random_xgb.fit(X_train, y_train)

print("Best xgboost params :", random_xgb.best_params_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best xgboost params : {'model__subsample': 1, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1}


Adaboost

In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

In [35]:
ada_pipeline = Pipeline([
		('smote', SMOTE(sampling_strategy=0.1, random_state=42)),
		('model', AdaBoostClassifier(
					estimator=DecisionTreeClassifier(), random_state=42 
		 ))
])

In [36]:
param_grid_ada = {
		 'model__n_estimators':[20,100,200],
		 'model__learning_rate':[0.1,0.5,1.0],
		 'model__estimator__max_depth':[1,2,3]
 }

In [41]:
 #RandomizedSearch
random_ada = RandomizedSearchCV(
		 ada_pipeline ,
		 param_distributions = param_grid_ada, 
		 n_iter=10,
		 scoring='f1',
		 cv=3,
		 verbose=1,
		 n_jobs=-1,
		 random_state=42,
 )
random_ada.fit(X_train,y_train)
 
print("Best adaboost params:", random_ada.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best adaboost params: {'model__n_estimators': 20, 'model__learning_rate': 0.1, 'model__estimator__max_depth': 2}


In [45]:
best_xgb = random_xgb.best_estimator_
best_ada = random_ada.best_estimator_
print("best estimator xgboost: ", best_xgb , "best estimator adaboost",best_ada )
y_pred_xgb = best_xgb.predict(X_test)
y_pred_ada = best_ada.predict(X_test)


best estimator xgboost:  Pipeline(steps=[('smote', SMOTE(random_state=42, sampling_strategy=0.1)),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=1, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.01,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=7, max_leaves=None,
                               min_child_w

In [43]:
from sklearn.metrics import classification_report

print("XGBoost Tuned:\n", classification_report(y_test, y_pred_xgb))
print("AdaBoost Tuned:\n", classification_report(y_test, y_pred_ada))

XGBoost Tuned:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.75      0.82      0.78        98

    accuracy                           1.00     56962
   macro avg       0.88      0.91      0.89     56962
weighted avg       1.00      1.00      1.00     56962

AdaBoost Tuned:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.47      0.88      0.61        98

    accuracy                           1.00     56962
   macro avg       0.73      0.94      0.81     56962
weighted avg       1.00      1.00      1.00     56962



Adaboost a trés bien améliorer ses résultats il a passer de 0.11(11%) a 0.47(47%) ce qui montre une réduction importante des faux positifs

pour le xgboost on remarque que la precision a diminuer a cause de fine_tuning qu'on a fait pour améliorer le F1 
Quand le recall augement on aura une meilleur detection mais la précision abaisse (ici ajoute explique d'une facon tres simple )

Quand le recall augmente, le modèle essaie de détecter le maximum de fraudes possible. Pour ne pas rater de cas frauduleux, 
il devient plus “sensible” et classe plus de transactions comme frauduleuses.

Après fine-tuning, XGBoost est plus performant qu’AdaBoost grâce à un meilleur équilibre entre précision et rappel, offrant un F1-score plus élevé et moins de fausses alertes.